In [13]:
# ─── Silver transformation for remaining 6 tables ──────────
# Schemas verified against the actual bronze_* tables in lh_bronze.dbo.
# Each table follows the same core pattern:
#   read → snake_case → widen keys to long → null handling → dedupe → write + OPTIMIZE

def snake_case_columns(df):
    """Standardize all column names to snake_case."""
    return df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

def dedupe_on_key(df, key_cols, order_col="_ingested_at"):
    """Keep the latest row per key (supports composite keys)."""
    if isinstance(key_cols, str):
        key_cols = [key_cols]
    w = Window.partitionBy(*key_cols).orderBy(F.col(order_col).desc())
    return (df
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

def to_silver(df):
    """Swap Bronze audit columns for Silver audit column."""
    return (df
        .drop("_ingested_at", "_source_file")
        .withColumn("_silver_loaded_at", F.current_timestamp())
    )

# ─── date dimension ────────────────────────────────────────
# Date already has good types; just standardize names and dedupe
df = spark.table("lh_bronze.dbo.bronze_date")
df = snake_case_columns(df)
df = dedupe_on_key(df, "datekey")
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_date")
spark.sql("OPTIMIZE silver_date")
print(f"✅ silver_date: {spark.table('silver_date').count():,} rows")

# ─── store dimension ───────────────────────────────────────
df = spark.table("lh_bronze.dbo.bronze_store")
df = snake_case_columns(df)
df = (df
    .withColumn("storekey",    F.col("storekey").cast(LongType()))
    .withColumn("geoareakey",  F.col("geoareakey").cast(LongType()))
    # Real null handling based on actual columns
    .withColumn("countryname", F.coalesce(F.col("countryname"), F.lit("UNKNOWN")))
    .withColumn("state",       F.coalesce(F.col("state"), F.lit("UNKNOWN")))
    .withColumn("status",      F.coalesce(F.col("status"), F.lit("Unknown")))
)
df = dedupe_on_key(df, "storekey")
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_store")
spark.sql("OPTIMIZE silver_store")
print(f"✅ silver_store: {spark.table('silver_store').count():,} rows")

# ─── product dimension ─────────────────────────────────────
df = spark.table("lh_bronze.dbo.bronze_product")
df = snake_case_columns(df)
df = (df
    .withColumn("productkey",     F.col("productkey").cast(LongType()))
    .withColumn("categorykey",    F.col("categorykey").cast(LongType()))
    .withColumn("subcategorykey", F.col("subcategorykey").cast(LongType()))
    .withColumn("brand",          F.coalesce(F.col("brand"), F.lit("UNKNOWN")))
    .withColumn("color",          F.coalesce(F.col("color"), F.lit("UNKNOWN")))
)
df = dedupe_on_key(df, "productkey")
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_product")
spark.sql("OPTIMIZE silver_product")
print(f"✅ silver_product: {spark.table('silver_product').count():,} rows")

# ─── orders header ─────────────────────────────────────────
df = spark.table("lh_bronze.dbo.bronze_orders")
df = snake_case_columns(df)
df = (df
    .withColumn("orderkey",    F.col("orderkey").cast(LongType()))
    .withColumn("customerkey", F.col("customerkey").cast(LongType()))
    .withColumn("storekey",    F.col("storekey").cast(LongType()))
)
df = dedupe_on_key(df, "orderkey")
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_orders")
spark.sql("OPTIMIZE silver_orders")
print(f"✅ silver_orders: {spark.table('silver_orders').count():,} rows")

# ─── order rows (line-item grain) ──────────────────────────
df = spark.table("lh_bronze.dbo.bronze_orderrows")
df = snake_case_columns(df)
df = (df
    .withColumn("orderkey",   F.col("orderkey").cast(LongType()))
    .withColumn("productkey", F.col("productkey").cast(LongType()))
)
df = dedupe_on_key(df, ["orderkey", "linenumber"])
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_orderrows")
spark.sql("OPTIMIZE silver_orderrows ZORDER BY (orderkey)")
print(f"✅ silver_orderrows: {spark.table('silver_orderrows').count():,} rows")

# ─── sales fact ────────────────────────────────────────────
df = spark.table("lh_bronze.dbo.bronze_sales")
df = snake_case_columns(df)
df = (df
    .withColumn("orderkey",    F.col("orderkey").cast(LongType()))
    .withColumn("customerkey", F.col("customerkey").cast(LongType()))
    .withColumn("storekey",    F.col("storekey").cast(LongType()))
    .withColumn("productkey",  F.col("productkey").cast(LongType()))
)
df = dedupe_on_key(df, ["orderkey", "linenumber"])
df = to_silver(df)
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_sales")
spark.sql("OPTIMIZE silver_sales ZORDER BY (customerkey, orderdate)")
print(f"✅ silver_sales: {spark.table('silver_sales').count():,} rows")

print("\n🎉 Phase 2 complete — all 7 Silver tables written and optimized.")

StatementMeta(, 55559020-3803-43a3-8465-be462fe126ff, 21, Finished, Available, Finished, False)

✅ silver_date: 4,018 rows
✅ silver_store: 74 rows
✅ silver_product: 2,517 rows
✅ silver_orders: 93,470 rows
✅ silver_orderrows: 223,974 rows
✅ silver_sales: 223,974 rows

🎉 Phase 2 complete — all 7 Silver tables written and optimized.


In [14]:
# Delta Time Travel — runs in Spark SQL (notebook)
print("=== silver_customer history ===")
spark.sql("DESCRIBE HISTORY silver_customer").show(truncate=False)

print("\n=== Version 0 row count ===")
v0 = spark.sql("SELECT COUNT(*) AS rc FROM silver_customer VERSION AS OF 0").collect()[0]["rc"]
print(f"silver_customer at version 0: {v0:,} rows")

print("\n=== Current version row count ===")
current = spark.sql("SELECT COUNT(*) AS rc FROM silver_customer").collect()[0]["rc"]
print(f"silver_customer current:      {current:,} rows")

StatementMeta(, 55559020-3803-43a3-8465-be462fe126ff, 22, Finished, Available, Finished, False)

=== silver_customer history ===
+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |notebook|clusterId|readVersion|isolationLevel   |isBlindAppend|ope